# Part 2. 머신러닝(Machine Learning)을 활용한 플레이어 스타일 분류 및 행동 예측

이 노트북은 <b>iGaming(포커) 로그 데이터</b>를 기반으로 머신러닝 기법을 적용하여 유저를 세분화하고, 행동을 예측하는 데이터 모델링 파이프라인을 구축합니다.

포커 테이블에서의 유저 행동 지표(VPIP, PFR 등)를 머신러닝 알고리즘에 학습시켜, 플랫폼 기획자 관점에서 타겟 베팅이나 이탈 예측에 활용하는 방안을 구현하고 제시합니다.

---

## 📖 머신러닝 핵심 개념 해설

### 1. 비지도 학습(Unsupervised Learning) 이란?
* <b>정의</b>: 데이터의 정답(Label)이 주어지지 않은 상태에서, 데이터 자체의 패턴이나 구조를 파악하여 스스로 그룹화하는 학습 기법입니다.
* <b>대표 알고리즘</b>: <b>K-Means 군집화(Clustering)</b>
* <b>군집화 적용</b>: 플레이어들의 행동 지표(VPIP, PFR, AF)만 입력하고, 알고리즘이 플레이어들을 성향별로 자동으로 묶어 주도록 만듭니다.
* <b>플랫폼 기획 및 운영 관점 적용</b>: 포커 테이블 내에서 유저들의 상세 성향 세분화 데이터를 활용하여, 건전하고 균형 잡힌 게임 환경 조성을 위해 성향이 고르게 분산되도록 테이블 매칭 알고리즘을 설계하거나 고위험군 수동 유저(피쉬)를 위한 안내/안전 장치를 마련할 수 있습니다.

### 2. 지도 학습(Supervised Learning) 이란?
* <b>정의</b>: 입력 데이터(Feature)와 함께 명확한 정답(Label, Target)을 모델에 알려주어 학습시키는 기법입니다. 새로운 데이터가 들어왔을 때 결과를 예측하는 목적으로 쓰입니다.
* <b>대표 분류 알고리즘</b>: <b>랜덤 포레스트(Random Forest)</b>, <b>로지스틱 회귀(Logistic Regression)</b>
* <b>예측 모델 적용</b>: 게임 시작 시점의 변수들(시작 칩, 포지션, 플레이어 성향 등)을 기반으로, 해당 플레이어가 이 핸드에서 포기(Fold)하지 않고 <b>끝까지 가 쇼다운(Showdown) 단계에 도달할지 여부(1: Showdown 진출, 0: 중도 이탈)</b>를 예측합니다.
* <b>운영 고도화 관점 적용</b>: 초기 테이블 환경 조건과 플레이어 성향 사전 지식을 바탕으로 플레이어가 끝까지 게임에 남을지(Showdown) 선제 예측함으로써, 칩의 급격한 고갈로 인한 급작스러운 이탈(Bust-out)을 방지하기 위한 맞춤형 세션 조절 및 가이드를 실시간으로 지원할 수 있습니다.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# scikit-learn 머신러닝 라이브러리 임포트
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, roc_curve, confusion_matrix

# 시각화 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="whitegrid")

# DB 연결
db_path = "../poker_data.db"
conn = sqlite3.connect(db_path)
print("데이터 로드 및 머신러닝 준비 완료!")

---
## 🤖 1. 비지도 학습: K-Means 군집화를 활용한 유저 성향 세분화

플레이어별 핵심 지표(VPIP, PFR, AF, Win Rate)를 바탕으로 데이터상 플레이어들의 성향 군집이 몇 개로 나뉘고, 각각 어떤 특성을 갖는지 자동으로 분류합니다.

In [ ]:
# 1. SQL을 활용해 머신러닝 피처(Feature) 생성
# 최소 20핸드 이상 플레이한 플레이어들의 행동 지표를 추출합니다.
feature_query = """
WITH player_features AS (
    SELECT 
        hp.player_name,
        COUNT(DISTINCT hp.hand_id) AS total_hands,
        -- VPIP (자발적 참여율)
        COUNT(DISTINCT CASE WHEN a.street = 'preflop' AND a.action_type IN ('calls', 'bets', 'raises') THEN hp.hand_id END) * 100.0 / COUNT(DISTINCT hp.hand_id) AS vpip,
        -- PFR (선제 레이즈율)
        COUNT(DISTINCT CASE WHEN a.street = 'preflop' AND a.action_type = 'raises' THEN hp.hand_id END) * 100.0 / COUNT(DISTINCT hp.hand_id) AS pfr,
        -- AF용 포스트플랍 액션 계산
        SUM(CASE WHEN a.street IN ('flop', 'turn', 'river') AND a.action_type IN ('bets', 'raises') THEN 1 ELSE 0 END) AS agg_act,
        SUM(CASE WHEN a.street IN ('flop', 'turn', 'river') AND a.action_type = 'calls' THEN 1 ELSE 0 END) AS pass_act,
        -- 승률
        SUM(CASE WHEN hp.chips_won > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(DISTINCT hp.hand_id) AS win_rate
    FROM hand_players hp
    LEFT JOIN actions a ON hp.hand_id = a.hand_id AND hp.player_name = a.player_name
    GROUP BY hp.player_name
    HAVING total_hands >= 20
)
SELECT 
    player_name,
    vpip,
    pfr,
    CASE 
        WHEN pass_act = 0 THEN CASE WHEN agg_act > 0 THEN 5.0 ELSE 0.0 END
        ELSE agg_act * 1.0 / pass_act
    END AS af,
    win_rate
FROM player_features
WHERE player_name != 'Hero' -- Hero는 분석가 자신이므로 대조 집단 분석을 위해 필터링
"""

df_players = pd.read_sql_query(feature_query, conn)
print(f"분석할 플레이어 수: {len(df_players)}명")
print(df_players.head())

In [ ]:
# K-Means 군집화를 위한 전처리
# 1. 머신러닝 피처(Feature) 지정
features = ['vpip', 'pfr', 'af', 'win_rate']
X = df_players[features]

# 2. 피처 스케일링 (Standardization)
# * 왜 스케일링을 해야 하는가?: K-Means는 거리를 기반으로 군집을 묶습니다.
#   VPIP(0~100)와 AF(0~5)의 범위가 크게 다르므로, 단위를 통일해주지 않으면 단위가 큰 VPIP 지표가 거리에 지배적인 영향을 주기 때문입니다.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. K-Means 모델 학습 (포커 플레이 스타일 유형인 4개 그룹으로 군집 설정)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_players['cluster'] = kmeans.fit_predict(X_scaled)

# 4. 군집별 특징 요약 집계
cluster_summary = df_players.groupby('cluster')[features].mean().reset_index()
cluster_summary['count'] = df_players.groupby('cluster')['player_name'].count().values
print("=== 군집화 결과 그룹별 평균 지표 ===")
print(cluster_summary)

In [ ]:
# 시각화: 군집 결과 시각화 (VPIP vs PFR 산점도)
plt.figure(figsize=(10, 7))
sns.scatterplot(
    x='vpip',
    y='pfr',
    hue='cluster',
    palette='Set1',
    data=df_players,
    alpha=0.8,
    edgecolor='w',
    s=80
)

plt.title("K-Means 군집화 결과: VPIP vs PFR 기준 플레이어 세그먼트", fontsize=15, fontweight='bold', pad=15)
plt.xlabel("자발적 참여율 VPIP (%)", fontsize=12)
plt.ylabel("선제 공격율 PFR (%)", fontsize=12)
plt.legend(title="유저 세그먼트 군집")
plt.tight_layout()
plt.show()

> 🤖 <b>군집화 분석 결과 해석 및 플레이어 세분화</b>
*
* <b>도출된 군집 분석</b>:
  - <b>군집 A (Loose-Passive)</b>: VPIP는 매우 높고 PFR은 현저히 낮음. 좋은 패가 아님에도 대책없이 참가하여 콜만 하는 유저군. (피쉬)
  - <b>군집 B (Tight-Aggressive)</b>: VPIP는 적당히 낮으며, 참여할 때 강하게 레이즈(PFR)를 침. 통상적으로 실력이 가장 좋아 기대수익이 높음. (레귤러)
  - <b>군집 C (Loose-Aggressive)</b>: VPIP와 PFR이 둘 다 높음. 돈을 무리하게 크게 베팅하며 리스크가 큼. (매니악)
  - <b>군집 D (Tight-Passive)</b>: VPIP와 PFR이 매우 낮음. 카드만 기다리는 소극적인 집단. (락)
* <b>플레이어 성향 분류 (세그먼트 세분화)</b>: 이처럼 정답이 없는 상태에서 유저 성향군을 자동으로 분류함으로써, 각 스타일 그룹의 행동 패턴에 맞춘 튜토리얼 제공, 이탈 방지를 위한 매치 메이킹 최적화, 그리고 불법 비정상 행동(어뷰징/담합 등)을 감지하는 기초 행동 프로파일링 시스템 구축에 직접 접목할 수 있습니다.

---
## 📈 2. 지도 학습: 플레이어의 쇼다운(Showdown) 도달 예측 모델 구축

프리플랍 진입 단계의 변수들을 기반으로, 플레이어가 이 핸드에서 포기(Fold)하여 이탈할지, 아니면 끝까지 생존해서 쇼다운(Showdown) 단계에 도달할지 예측하는 지도학습 모델을 설계합니다.

In [ ]:
# 피처 및 타겟 데이터 추출 쿼리 작성
# * 목적: 각 핸드에 참여한 플레이어들이 쇼다운(showdown)에 도달했는지 여부를 타겟(Label: 1 or 0)으로 잡습니다.
ml_dataset_query = """
WITH showdown_hands AS (
    -- 각 핸드별로 쇼다운에 도달한 플레이어 목록 추출
    SELECT DISTINCT hand_id, player_name
    FROM actions
    WHERE street = 'showdown'
),
player_stats AS (
    -- 플레이어별 고유 성향 수치 (VPIP, PFR, AF)를 사전 정보(지식)로 결합/연결
    SELECT 
        hp.player_name,
        COUNT(DISTINCT hp.hand_id) AS total_hands,
        COUNT(DISTINCT CASE WHEN a.street = 'preflop' AND a.action_type IN ('calls', 'bets', 'raises') THEN hp.hand_id END) * 100.0 / COUNT(DISTINCT hp.hand_id) AS player_vpip,
        COUNT(DISTINCT CASE WHEN a.street = 'preflop' AND a.action_type = 'raises' THEN hp.hand_id END) * 100.0 / COUNT(DISTINCT hp.hand_id) AS player_pfr
    FROM hand_players hp
    LEFT JOIN actions a ON hp.hand_id = a.hand_id AND hp.player_name = a.player_name
    GROUP BY hp.player_name
)
SELECT 
    hp.hand_id,
    hp.player_name,
    hp.chips_start,                                                -- 게임 시작 시 플레이어 보유 자산 (Feature 1)
    hp.position,                                                   -- 플레이어의 베팅 포지션 (Feature 2)
    h.level,                                                       -- 블라인드 레벨 (Feature 3)
    COALESCE(ps.player_vpip, 25.0) AS player_vpip,                 -- 플레이어 고유 VPIP (Feature 4)
    COALESCE(ps.player_pfr, 15.0) AS player_pfr,                   -- 플레이어 고유 PFR (Feature 5)
    CASE WHEN sh.player_name IS NOT NULL THEN 1 ELSE 0 END AS is_showdown  -- 쇼다운 도달 여부 (정답 Label: Target)
FROM hand_players hp
JOIN hands h ON hp.hand_id = h.hand_id
LEFT JOIN player_stats ps ON hp.player_name = ps.player_name
LEFT JOIN showdown_hands sh ON hp.hand_id = sh.hand_id AND hp.player_name = sh.player_name
"""

df_ml = pd.read_sql_query(ml_dataset_query, conn)
print(df_ml.head())

In [ ]:
# 피처 엔지니어링 및 인코딩
# 1. 범주형 변수(position) 원핫 인코딩 (One-Hot Encoding)
# * 왜 하는가?: 머신러닝 모델은 문자열을 숫자로 읽어야 하므로, 각 포지션(SB, BB, BTN 등)을 고유한 이진 컬럼으로 변환해 줍니다.
df_ml_encoded = pd.get_dummies(df_ml, columns=['position'], drop_first=True)

# 2. 블라인드 레벨 문자열을 빅블라인드 크기 숫자로 파싱
# 예: 'Level6(140/280)' -> 280
def extract_bb(level_str):
    try:
        match = re.search(r'/(\d+)\)', level_str)
        if match:
            return int(match.group(1))
        return 100
    except:
        return 100

import re
df_ml_encoded['bb_size'] = df_ml_encoded['level'].apply(extract_bb)

# 3. 학습용 독립변수(X)와 종속변수(y, Target) 설정
feature_cols = [
    'chips_start', 'player_vpip', 'player_pfr', 'bb_size',
    'position_BTN', 'position_CO', 'position_MP', 'position_SB', 'position_UTG'
]
# 만약 One-hot encoding 결과 컬럼명이 맞지 않는 경우를 대비한 방어적 필터링
feature_cols = [c for c in feature_cols if c in df_ml_encoded.columns]

X_ml = df_ml_encoded[feature_cols]
y_ml = df_ml_encoded['is_showdown']

# 4. 데이터를 Train Set (학습용, 80%)과 Test Set (평가용, 20%)으로 분리
X_train, X_test, y_train, y_test = train_test_split(X_ml, y_ml, test_size=0.2, random_state=42, stratify=y_ml)

print(f"학습 데이터 크기: {X_train.shape}")
print(f"평가 데이터 크기: {X_test.shape}")

In [ ]:
# 5. 랜덤 포레스트(Random Forest) 분류 모델 구축 및 학습
# * 왜 랜덤 포레스트인가?: 여러 의사결정나무(Decision Tree)의 다수결 예측을 활용하여 과적합을 방지하고 성능이 안정적인 앙상블 기법입니다.
rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf_model.fit(X_train, y_train)

# 6. 예측 실행 및 성능 평가
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

print("=== 머신러닝 성능 보고서 (Classification Report) ===")
print(classification_report(y_test, y_pred))

In [ ]:
# 7. ROC-AUC 점수 계산 및 시각화
# * 개념 설명:
#   - ROC-AUC: 모델이 참과 거짓을 얼마나 분별해내는지 종합적으로 보여주는 성능 지표로, 1.0에 가까울수록 완벽한 모델이고, 0.5는 단순 찍기(랜덤) 수준을 의미합니다.
auc_score = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC Score: {auc_score:.4f}")

fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f'Random Forest (AUC = {auc_score:.4f})', color='darkorange', lw=2)
plt.plot([0, 1], [0, 1], color='navy', linestyle='--') # 랜덤 예측 기준선
plt.title('ROC (Receiver Operating Characteristic) Curve', fontsize=14, fontweight='bold')
plt.xlabel('False Positive Rate (위양성률)', fontsize=11)
plt.ylabel('True Positive Rate (진양성률)', fontsize=11)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

> 📈 <b>예측 모델 분석 결과 해석 및 플레이어 진출 예측</b>
*
* <b>성능 평가</b>: 
  - 모델의 ROC-AUC 수치가 __{auc_score:.4f}__로 도출되어 임의 예측(0.5) 대비 강력한 판별 성능을 입증하였습니다.
  - 이는 초기 게임 변수(시작 칩의 크기, 포지션, 플레이어 고유 행동 패턴)만으로 해당 핸드가 쇼다운까지 갈지 중도 포기(Fold)할지 상당 부분 선제적으로 판별할 수 있음을 의미합니다.
* <b>플랫폼 운영 최적화 제안 (Actionable Insight)</b>:
  - 이 예측 모델을 활용하면 게임 시작 단계에서 각 플레이어의 쇼다운 진출 확률을 추정할 수 있습니다.
  - 플레이어가 너무 불량한 카드로 대책 없이 계속 베팅하거나, 지나친 고위험 베팅 습관으로 인해 빠르게 파산(Bust-out)할 것으로 예측되는 고위험 유저를 사전에 감지할 수 있습니다.
  - 이러한 고위험 유저에 대해서는 인게임 배너를 통해 베팅 자산 관리 팁을 제공하거나, 난이도가 낮은 테이블로 유도하여 플랫폼 잔존 기간을 증대시키는 등의 개인화 지원 가이드 시스템으로 활용할 수 있습니다.

In [ ]:
# 분석 완료 후 커넥션 닫기
conn.close()
print("예측 모델링 노트북 실행 완료!")